In [1]:
"""
Hardware QGT Proxy — Figures (project-style)
============================================
Input:  qgt_echo_results.json
Output: fig_echo1_delta_scan.pdf   — three-panel δ-scan
        fig_echo2_gphi_scan.pdf    — standalone g_φφ(θ)

Style matches the rest of the project:
  - Times New Roman (falls back to STIX Two Text)
  - No titles, no grids
  - Bold (a)/(b)/(c) panel labels in upper-left
  - PRX rcParams (inward ticks, top/right ticks on, axes.linewidth=0.8)
  - Palette preserved: HW=red, sim=blue, analytical=green, fit=purple
"""

import json, os, math
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FuncFormatter, FixedLocator

# ============================================================
# Paths
# ============================================================
ECHO_JSON = "/Users/nandan/Desktop/CTCs/IBM/qgt_echo_results.json"
OUT_DIR   = "/Users/nandan/Desktop/CTCs/IBM"

# ============================================================
# Font selection — real Times if available, else STIX Two Text
# ============================================================
installed = {f.name for f in fm.fontManager.ttflist}
if "Times New Roman" in installed:
    SERIF_STACK = ["Times New Roman"]
elif "Times" in installed:
    SERIF_STACK = ["Times"]
else:
    SERIF_STACK = ["STIX Two Text", "STIXGeneral", "DejaVu Serif"]

mpl.rcParams.update({
    "font.family":         "serif",
    "font.serif":          SERIF_STACK,
    "mathtext.fontset":    "stix",
    "font.size":           11,
    "axes.labelsize":      12,
    "axes.titlesize":      12,
    "xtick.labelsize":     10.5,
    "ytick.labelsize":     10.5,
    "legend.fontsize":     9,
    "axes.linewidth":      0.8,
    "xtick.major.width":   0.8,
    "ytick.major.width":   0.8,
    "xtick.minor.width":   0.6,
    "ytick.minor.width":   0.6,
    "xtick.major.size":    3.5,
    "ytick.major.size":    3.5,
    "xtick.minor.size":    2.0,
    "ytick.minor.size":    2.0,
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.top":           True,
    "ytick.right":         True,
    "pdf.fonttype":        42,
    "ps.fonttype":         42,
})

# Project palette (kept identical to previous echo figures)
C_HW   = "#C0392B"   # hardware:   red
C_SIM  = "#2980B9"   # simulator:  blue
C_ANAL = "#27AE60"   # analytical: green
C_FIT  = "#8E44AD"   # HW fit:     purple

# ============================================================
# Helpers
# ============================================================
def pi_fmt(val, _pos):
    n = val / math.pi
    if abs(n) < 0.01:        return "0"
    if abs(n - 0.25) < 0.01: return r"$\pi/4$"
    if abs(n - 0.5)  < 0.01: return r"$\pi/2$"
    if abs(n - 0.75) < 0.01: return r"$3\pi/4$"
    if abs(n - 1)    < 0.01: return r"$\pi$"
    return f"${n:.2f}\\pi$"


def panel_label(ax, label, x=-0.20, y=1.04):
    """Bold (a)/(b)/(c) tag in upper-left, outside the plot box."""
    ax.text(x, y, label,
            transform=ax.transAxes,
            fontsize=15, fontweight="bold", va="bottom", ha="left")


def P00_model(d, g):
    return 1.0 - 4.0 * g * np.sin(d / 2) ** 2


# ============================================================
# Load data
# ============================================================
with open(ECHO_JSON) as f:
    D = json.load(f)

meta   = D["metadata"]
analyt = D["analytical"]
hw     = D["hardware"]
shots  = meta["shots"]

deltas      = np.array(hw["g_tt"]["deltas"])
P00_hw      = np.array(hw["g_tt"]["P00_hw"])
P00_lo      = np.array(hw["g_tt"]["P00_lo"])
P00_hi      = np.array(hw["g_tt"]["P00_hi"])
P00_sim     = np.array(hw["g_tt"]["P00_sim"])
g_tt_fit    = hw["g_tt"]["g_fit"]
g_tt_bs_lo  = hw["g_tt"]["g_bs_lo"]
g_tt_bs_hi  = hw["g_tt"]["g_bs_hi"]
g_tt_exact  = analyt["g_tt_exact"]
bs_g        = np.array(hw["g_tt"]["bootstrap_g"])

thetas_scan = np.array(hw["g_pp"]["thetas"])
g_pp_hw     = np.array(hw["g_pp"]["g_pp_hw"])
g_pp_lo     = np.array(hw["g_pp"]["g_pp_lo"])
g_pp_hi     = np.array(hw["g_pp"]["g_pp_hi"])
g_pp_anal   = np.array(hw["g_pp"]["g_pp_analytical"])

os.makedirs(OUT_DIR, exist_ok=True)

delta_fine = np.linspace(0, max(deltas) * 1.08, 300)

# Per-δ g_θθ estimates (for panel b)
g_hw_per_d  = (1 - P00_hw)  / (4 * np.sin(deltas / 2) ** 2)
g_lo_per_d  = (1 - P00_hi)  / (4 * np.sin(deltas / 2) ** 2)
g_hi_per_d  = (1 - P00_lo)  / (4 * np.sin(deltas / 2) ** 2)
g_sim_per_d = (1 - P00_sim) / (4 * np.sin(deltas / 2) ** 2)


# =============================================================================
#  FIG 1 — three-panel δ-scan
# =============================================================================
fig1, axes1 = plt.subplots(1, 3, figsize=(15, 4.6))

# ----- (a) Echo P_00 vs δ ---------------------------------------------------
ax = axes1[0]
ax.plot(delta_fine, P00_model(delta_fine, g_tt_exact),
        color=C_ANAL, lw=2.0, ls="--",
        label=f"Analytical ($g={g_tt_exact:.5f}$)", zorder=3)
ax.plot(delta_fine, P00_model(delta_fine, g_tt_fit),
        color=C_FIT, lw=2.0, ls=":",
        label=f"HW fit ($g={g_tt_fit:.5f}$)", zorder=4)
ax.scatter(deltas, P00_sim, s=50, marker="s",
           facecolors="none", edgecolors=C_SIM, linewidths=1.5,
           label="Simulator", zorder=5)
ax.errorbar(deltas, P00_hw,
            yerr=[P00_hw - P00_lo, P00_hi - P00_hw],
            fmt="o", color=C_HW, ms=6, lw=1.5, capsize=4, capthick=1.2,
            label=f"Hardware ({shots:,} shots)", zorder=6)
ax.set_xlabel(r"$\delta$")
ax.set_ylabel(r"$P_{00}(\delta) = |\langle\Psi|\Psi_\delta\rangle|^2$")
ax.legend(framealpha=0.9, loc="lower left")
panel_label(ax, "(a)")

# ----- (b) Per-δ g_θθ estimate (should be flat) -----------------------------
ax = axes1[1]
ax.axhline(g_tt_exact, color=C_ANAL, lw=2.0, ls="--",
           label=f"Analytical: {g_tt_exact:.5f}", zorder=3)
ax.axhspan(g_tt_bs_lo, g_tt_bs_hi, alpha=0.18, color=C_FIT,
           label=f"HW 95% CI [{g_tt_bs_lo:.5f}, {g_tt_bs_hi:.5f}]")
ax.axhline(g_tt_fit, color=C_FIT, lw=2.0, ls=":",
           label=f"HW fit: {g_tt_fit:.5f}", zorder=4)
ax.scatter(deltas, g_sim_per_d, s=50, marker="s",
           facecolors="none", edgecolors=C_SIM, linewidths=1.5,
           label="Simulator point-estimate", zorder=5)
ax.errorbar(deltas, g_hw_per_d,
            yerr=[g_hw_per_d - g_lo_per_d, g_hi_per_d - g_hw_per_d],
            fmt="o", color=C_HW, ms=6, lw=1.5, capsize=4,
            label="Hardware", zorder=6)
ax.set_xlabel(r"$\delta$")
ax.set_ylabel(r"$(1-P_{00})\,/\,[4\sin^2(\delta/2)]$")
ax.legend(framealpha=0.9)
panel_label(ax, "(b)")

# ----- (c) Bootstrap distribution of g --------------------------------------
ax = axes1[2]
ax.hist(bs_g, bins=40, color=C_FIT, alpha=0.75, edgecolor="white", lw=0.5)
ax.axvline(g_tt_exact, color=C_ANAL, lw=2.0, ls="--",
           label=f"Analytical: {g_tt_exact:.5f}")
ax.axvline(g_tt_fit, color=C_FIT, lw=2.0,
           label=f"HW fit: {g_tt_fit:.5f}")
ax.axvline(g_tt_bs_lo, color=C_HW, lw=1.2, ls=":", alpha=0.8)
ax.axvline(g_tt_bs_hi, color=C_HW, lw=1.2, ls=":", alpha=0.8,
           label="95% CI")
ax.set_xlabel(r"$g_{\theta\theta}$")
ax.set_ylabel("Bootstrap count")
ax.legend(framealpha=0.9)
panel_label(ax, "(c)")

fig1.tight_layout()
fig1.savefig(os.path.join(OUT_DIR, "echo1_delta_scan.pdf"),
             bbox_inches="tight", dpi=300)
fig1.savefig(os.path.join(OUT_DIR, "echo1_delta_scan.png"),
             bbox_inches="tight", dpi=300)
plt.close(fig1)
print("[✓] fig_echo1_delta_scan.pdf")


# =============================================================================
#  FIG 2 — standalone g_φφ(θ) scan
# =============================================================================
fig2, ax = plt.subplots(figsize=(5.5, 4.6))

ax.plot(thetas_scan, g_pp_anal, color=C_ANAL, lw=2.0, ls="-",
        label=r"Analytical $(1-\langle\sigma_z\rangle^2_{C,\theta})/4$",
        zorder=4)
ax.errorbar(thetas_scan, g_pp_hw,
            yerr=[g_pp_hw - g_pp_lo, g_pp_hi - g_pp_hw],
            fmt="o", color=C_HW, ms=6, lw=1.5, capsize=4,
            label=rf"Hardware ($\delta\phi={meta['delta_phi_fixed']:.2f}$)",
            zorder=5)

ax.set_xlim(0, np.pi)
ax.xaxis.set_major_locator(FixedLocator([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]))
ax.xaxis.set_major_formatter(FuncFormatter(pi_fmt))
ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$g_{\phi\phi}(\theta)$")
ax.legend(framealpha=0.9)

fig2.tight_layout()
fig2.savefig(os.path.join(OUT_DIR, "echo2_gphi_scan.pdf"),
             bbox_inches="tight", dpi=300)
fig2.savefig(os.path.join(OUT_DIR, "_echo2_gphi_scan.png"),
             bbox_inches="tight", dpi=300)
plt.close(fig2)
print("[✓] fig_echo2_gphi_scan.pdf")

# =============================================================================
#  PAPER-READY NUMBERS
# =============================================================================
print(f"\n{'='*60}")
print("  PAPER-READY NUMBERS")
print(f"{'='*60}")
print(f"  Backend: {meta['backend']},  shots: {shots:,},  DD: {meta.get('dd_sequence','XX')}")
print(f"  g_θθ exact      = {g_tt_exact:.6f}")
print(f"  g_θθ HW fit     = {g_tt_fit:.6f} ± {hw['g_tt']['g_fit_std']:.6f}")
print(f"  g_θθ 95% CI     = [{g_tt_bs_lo:.6f}, {g_tt_bs_hi:.6f}]")
print(f"  Δg              = {g_tt_fit - g_tt_exact:+.6f}  "
      f"({100*abs(g_tt_fit-g_tt_exact)/g_tt_exact:.2f}% error)")

[✓] fig_echo1_delta_scan.pdf
[✓] fig_echo2_gphi_scan.pdf

  PAPER-READY NUMBERS
  Backend: ibm_torino,  shots: 10,000,  DD: XX
  g_θθ exact      = 0.235788
  g_θθ HW fit     = 0.242299 ± 0.001742
  g_θθ 95% CI     = [0.237161, 0.244042]
  Δg              = +0.006511  (2.76% error)


In [2]:
"""
Hardware QGT Proxy — combined three-panel figure
================================================
Input:  qgt_echo_results.json
Output: fig_qgt_echo_combined.pdf
        fig_qgt_echo_combined.png

Panels:
  (a) Echo survival probability P_00(delta)
  (b) Per-delta estimator of g_{theta theta}
  (c) Standalone g_{phi phi}(theta) scan

Design choices for a paper figure:
  - One double-column, three-panel figure
  - Compact legends and short axis labels; details belong in the caption/text
  - Times New Roman if available, else Times/STIX fallback
  - No titles, no grids, inward ticks, top/right ticks on
  - Embedded TrueType fonts in PDF
"""

import json
import math
import os
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import AutoMinorLocator, FuncFormatter, FixedLocator, MultipleLocator

# ============================================================
# Paths
# ============================================================
# Change these two lines for your local machine.
ECHO_JSON = "/Users/nandan/Desktop/CTCs/IBM/qgt_echo_results.json"
OUT_DIR   = "/Users/nandan/Desktop/CTCs/IBM"
OUT_STEM = "fig_qgt_echo_combined"

# ============================================================
# Font selection — real Times if available, else STIX fallback
# ============================================================
installed = {f.name for f in fm.fontManager.ttflist}
if "Times New Roman" in installed:
    SERIF_STACK = ["Times New Roman"]
elif "Times" in installed:
    SERIF_STACK = ["Times"]
else:
    SERIF_STACK = ["STIX Two Text", "STIXGeneral", "DejaVu Serif"]

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": SERIF_STACK,
    "mathtext.fontset": "stix",
    "font.size": 8.8,
    "axes.labelsize": 9.5,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8.4,
    "ytick.labelsize": 8.4,
    "legend.fontsize": 7.1,
    "axes.linewidth": 0.75,
    "xtick.major.width": 0.75,
    "ytick.major.width": 0.75,
    "xtick.minor.width": 0.55,
    "ytick.minor.width": 0.55,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "xtick.minor.size": 1.7,
    "ytick.minor.size": 1.7,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.grid": False,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Project palette, preserved from earlier figures.
C_HW = "#C0392B"      # hardware:   red
C_SIM = "#2980B9"     # simulator:  blue
C_ANAL = "#27AE60"    # analytical: green
C_FIT = "#8E44AD"     # HW fit:     purple

# ============================================================
# Helpers
# ============================================================
def pi_fmt(val, _pos):
    n = val / math.pi
    if abs(n) < 0.01:
        return "0"
    if abs(n - 0.25) < 0.01:
        return r"$\pi/4$"
    if abs(n - 0.5) < 0.01:
        return r"$\pi/2$"
    if abs(n - 0.75) < 0.01:
        return r"$3\pi/4$"
    if abs(n - 1) < 0.01:
        return r"$\pi$"
    return rf"${n:.2f}\pi$"


def panel_label(ax, label, x=-0.16, y=1.035):
    """Bold panel tag in the upper-left, just outside the plot box."""
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        fontsize=11.8,
        fontweight="bold",
        va="bottom",
        ha="left",
        clip_on=False,
    )


def P00_model(delta, g):
    return 1.0 - 4.0 * g * np.sin(delta / 2.0) ** 2


def beautify(ax):
    """Common axis polish for a PRX/PRL-style figure."""
    for spine in ax.spines.values():
        spine.set_linewidth(0.75)
    ax.tick_params(which="both", direction="in", top=True, right=True)
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))


def compact_legend(ax, **kwargs):
    defaults = dict(
        frameon=True,
        fancybox=False,
        framealpha=0.92,
        edgecolor="0.25",
        borderpad=0.32,
        handlelength=1.65,
        handletextpad=0.45,
        labelspacing=0.24,
        borderaxespad=0.35,
    )
    defaults.update(kwargs)
    leg = ax.legend(**defaults)
    if leg is not None:
        leg.get_frame().set_linewidth(0.55)
    return leg


# ============================================================
# Load data
# ============================================================
with open(ECHO_JSON, "r") as f:
    D = json.load(f)

meta = D["metadata"]
analyt = D["analytical"]
hw = D["hardware"]
shots = meta["shots"]

deltas = np.asarray(hw["g_tt"]["deltas"], dtype=float)
P00_hw = np.asarray(hw["g_tt"]["P00_hw"], dtype=float)
P00_lo = np.asarray(hw["g_tt"]["P00_lo"], dtype=float)
P00_hi = np.asarray(hw["g_tt"]["P00_hi"], dtype=float)
P00_sim = np.asarray(hw["g_tt"]["P00_sim"], dtype=float)

g_tt_fit = float(hw["g_tt"]["g_fit"])
g_tt_bs_lo = float(hw["g_tt"]["g_bs_lo"])
g_tt_bs_hi = float(hw["g_tt"]["g_bs_hi"])
g_tt_exact = float(analyt["g_tt_exact"])

thetas_scan = np.asarray(hw["g_pp"]["thetas"], dtype=float)
g_pp_hw = np.asarray(hw["g_pp"]["g_pp_hw"], dtype=float)
g_pp_lo = np.asarray(hw["g_pp"]["g_pp_lo"], dtype=float)
g_pp_hi = np.asarray(hw["g_pp"]["g_pp_hi"], dtype=float)
g_pp_anal = np.asarray(hw["g_pp"]["g_pp_analytical"], dtype=float)

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

delta_fine = np.linspace(0.0, max(deltas) * 1.06, 400)

# Per-delta estimates for panel (b).  Since g = (1 - P_00)/(4 sin^2(delta/2)),
# the lower/upper probability bounds map to upper/lower g bounds.
den = 4.0 * np.sin(deltas / 2.0) ** 2
g_hw_per_d = (1.0 - P00_hw) / den
g_lo_per_d = (1.0 - P00_hi) / den
g_hi_per_d = (1.0 - P00_lo) / den
g_sim_per_d = (1.0 - P00_sim) / den

# ============================================================
# Combined figure
# ============================================================
# Double-column physics-journal width.  Increase to (8.2, 2.85) if your target
# journal uses a wider two-column text block.
fig, axes = plt.subplots(1, 3, figsize=(7.35, 2.55), constrained_layout=False)
fig.subplots_adjust(left=0.075, right=0.995, bottom=0.24, top=0.915, wspace=0.36)

# ----- (a) Echo P_00 vs delta ----------------------------------------------
ax = axes[0]
ax.plot(
    delta_fine,
    P00_model(delta_fine, g_tt_exact),
    color=C_ANAL,
    lw=1.55,
    ls="--",
    label="Analytical",
    zorder=3,
)
ax.plot(
    delta_fine,
    P00_model(delta_fine, g_tt_fit),
    color=C_FIT,
    lw=1.55,
    ls=":",
    label="HW fit",
    zorder=4,
)
ax.scatter(
    deltas,
    P00_sim,
    s=22,
    marker="s",
    facecolors="white",
    edgecolors=C_SIM,
    linewidths=1.05,
    label="Simulator",
    zorder=5,
)
ax.errorbar(
    deltas,
    P00_hw,
    yerr=[P00_hw - P00_lo, P00_hi - P00_hw],
    fmt="o",
    color=C_HW,
    markerfacecolor=C_HW,
    markeredgecolor=C_HW,
    markersize=3.8,
    elinewidth=0.95,
    capsize=2.5,
    capthick=0.9,
    lw=0,
    label=f"Hardware",
    zorder=6,
)
ax.set_xlim(0.42, max(deltas) * 1.05)
ax.set_ylim(0.48, 0.98)
ax.set_xlabel(r"$\delta$")
ax.set_ylabel(r"$P_{00}$")
beautify(ax)
compact_legend(ax, loc="lower left")
panel_label(ax, "(a)")

# ----- (b) Per-delta g_theta theta estimate --------------------------------
ax = axes[1]
ax.axhline(g_tt_exact, color=C_ANAL, lw=1.55, ls="--", label="Analytical", zorder=3)
ax.axhspan(g_tt_bs_lo, g_tt_bs_hi, alpha=0.16, color=C_FIT, label="HW 95% CI", zorder=1)
ax.axhline(g_tt_fit, color=C_FIT, lw=1.55, ls=":", label="HW fit", zorder=4)
ax.scatter(
    deltas,
    g_sim_per_d,
    s=22,
    marker="s",
    facecolors="white",
    edgecolors=C_SIM,
    linewidths=1.05,
    label="Simulator",
    zorder=5,
)
ax.errorbar(
    deltas,
    g_hw_per_d,
    yerr=[g_hw_per_d - g_lo_per_d, g_hi_per_d - g_hw_per_d],
    fmt="o",
    color=C_HW,
    markerfacecolor=C_HW,
    markeredgecolor=C_HW,
    markersize=3.8,
    elinewidth=0.95,
    capsize=2.5,
    capthick=0.9,
    lw=0,
    label="Hardware",
    zorder=6,
)
ax.set_xlim(0.42, max(deltas) * 1.05)
# Tight but not cramped range around the estimates.
ymin = min(g_lo_per_d.min(), g_sim_per_d.min(), g_tt_bs_lo, g_tt_exact)
ymax = max(g_hi_per_d.max(), g_sim_per_d.max(), g_tt_bs_hi, g_tt_fit)
pad = 0.12 * (ymax - ymin)
ax.set_ylim(ymin - pad, ymax + pad)
ax.set_xlabel(r"$\delta$")
ax.set_ylabel(r"$\hat g_{\theta\theta}(\delta)$")
beautify(ax)
compact_legend(ax, loc="upper right")
panel_label(ax, "(b)")

# ----- (c) g_phi phi(theta) scan -------------------------------------------
ax = axes[2]
ax.plot(
    thetas_scan,
    g_pp_anal,
    color=C_ANAL,
    lw=1.65,
    ls="-",
    label="Analytical",
    zorder=4,
)
ax.errorbar(
    thetas_scan,
    g_pp_hw,
    yerr=[g_pp_hw - g_pp_lo, g_pp_hi - g_pp_hw],
    fmt="o",
    color=C_HW,
    markerfacecolor=C_HW,
    markeredgecolor=C_HW,
    markersize=3.8,
    elinewidth=0.95,
    capsize=2.5,
    capthick=0.9,
    lw=0,
    label=rf"Hardware, $\delta\phi={meta['delta_phi_fixed']:.1f}$",
    zorder=5,
)
ax.set_xlim(0, np.pi)
ax.xaxis.set_major_locator(FixedLocator([0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi]))
ax.xaxis.set_major_formatter(FuncFormatter(pi_fmt))
ax.xaxis.set_minor_locator(MultipleLocator(np.pi / 8))
ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$g_{\phi\phi}(\theta)$")
beautify(ax)
compact_legend(ax, loc="upper left")
panel_label(ax, "(c)")

pdf_path = Path(OUT_DIR) / f"{OUT_STEM}.pdf"
png_path = Path(OUT_DIR) / f"{OUT_STEM}.png"
fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.02)
fig.savefig(png_path, bbox_inches="tight", pad_inches=0.02)
plt.close(fig)

print(f"[✓] {pdf_path}")
print(f"[✓] {png_path}")
print("\nPAPER-READY NUMBERS")
print("-" * 60)
print(f"Backend: {meta['backend']}, shots: {shots:,}, DD: {meta.get('dd_sequence', 'XX')}")
print(f"g_theta theta exact  = {g_tt_exact:.6f}")
print(f"g_theta theta HW fit = {g_tt_fit:.6f} ± {hw['g_tt']['g_fit_std']:.6f}")
print(f"g_theta theta 95% CI = [{g_tt_bs_lo:.6f}, {g_tt_bs_hi:.6f}]")
print(f"Delta g              = {g_tt_fit - g_tt_exact:+.6f} "
      f"({100 * abs(g_tt_fit - g_tt_exact) / g_tt_exact:.2f}% error)")


[✓] /Users/nandan/Desktop/CTCs/IBM/fig_qgt_echo_combined.pdf
[✓] /Users/nandan/Desktop/CTCs/IBM/fig_qgt_echo_combined.png

PAPER-READY NUMBERS
------------------------------------------------------------
Backend: ibm_torino, shots: 10,000, DD: XX
g_theta theta exact  = 0.235788
g_theta theta HW fit = 0.242299 ± 0.001742
g_theta theta 95% CI = [0.237161, 0.244042]
Delta g              = +0.006511 (2.76% error)
